In [0]:
!pip install openai kneed hdbscan umap-learn python-igraph leidenalg python-dotenv
dbutils.library.restartPython()

In [0]:

import logging
import sys

logger = logging.getLogger()   # root logger
logger.setLevel(logging.INFO)

# Remove existing handlers (Databricks adds some silently)
for h in logger.handlers[:]:
    logger.removeHandler(h)

# Add explicit stdout handler
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setLevel(logging.INFO)

formatter = logging.Formatter(
    "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
stream_handler.setFormatter(formatter)

logger.addHandler(stream_handler)


In [0]:

from dotenv import load_dotenv
load_dotenv("/Workspace/Repos/morgan.wang@rci.rogers.ca/CCTS_Journey_Model/.env")


In [0]:

import os
print("API KEY SET:", bool(os.getenv("AZURE_OPENAI_API_KEY")))
print("ENDPOINT:", os.getenv("AZURE_OPENAI_ENDPOINT"))
print("DATA PATH:", os.getenv("DATA_PICKLE_PATH"))



In [0]:
from report_generation import (
    ConfigManager,
    InteractionAnalyzer,
    save_analysis_results,
)
data_config = ConfigManager.load_data_config()
print(data_config)

In [0]:

from levels_report_generation import levels_report_generation 
levels_report_generation()


In [0]:
from theme_driver_analysis import theme_driver_analysis
theme_driver_analysis()

In [0]:
from resolution_recommendation_analysis import resolution_recommendation_analysis
resolution_recommendation_analysis()

In [0]:
theme_dir = "/Workspace/Users/morgan.wang@rci.rogers.ca/share_workspeace/Levels_report_generation/Journey_2512_full" 
def load_theme_config() :
    """Load theme analysis configuration from environment variables."""
    data_folder = os.getenv("THEME_DATA_FOLDER", theme_dir)
    if not data_folder:
        raise ValueError("THEME_DATA_FOLDER environment variable is required")

    text_column = os.getenv("THEME_TEXT_COLUMN", "primary_complaint_issue")
    clustering_method = os.getenv("THEME_CLUSTERING_METHOD", "leiden")
    reduce_dimensions = os.getenv("THEME_REDUCE_DIMENSIONS", "False").strip().lower() in (
        "1",
        "true",
        "yes",
    )
    output_path = os.getenv("THEME_OUTPUT_PATH", "./theme_output/")
    norm = "True"
    # Clustering parameters
    clustering_params = {}
    if clustering_method == "leiden":
        clustering_params["k"] = int(os.getenv("THEME_LEIDEN_K", "30"))
        clustering_params["resolution_parameter"] = float(os.getenv("THEME_LEIDEN_RESOLUTION", "0.7"))
        clustering_params["use_snn"] = os.getenv("THEME_LEIDEN_USE_SNN", "True").strip().lower() in ("1", "true", "yes")
        clustering_params["metric"] = os.getenv("THEME_LEIDEN_METRIC", "cosine")
        clustering_params["random_state"] = int(os.getenv("THEME_LEIDEN_RANDOM_STATE", "42"))
    elif clustering_method =="kmeans":
        clustering_params["n_clusters"] = "None",
        clustering_params["n_clusters"] = "True"
    elif  clustering_method == "dbscan":
        clustering_params["min_cluster_size"] = "30"
        clustering_params["min_samples"] = "10"
        clustering_params["metrics"] = "euclidean"

    return {
        "data_folder": data_folder,
        "text_column": text_column,
        "clustering_method": clustering_method,
        "reduce_dimensions": reduce_dimensions,
        "output_path": output_path,
        "clustering_params": clustering_params,
        "norm": norm
    }


In [0]:
theme_config = load_theme_config()

In [0]:
theme_config

In [0]:

import sys
import json
from pathlib import Path
from typing import Any, Dict
from openai import AzureOpenAI
from ccts_theme_driver_analysis import ThemeAnalyzer
from report_generation import ConfigManager
def save_theme_results(results: Dict[str, Any], output_path: str) -> None:
    """Save a lightweight theme analysis summary to JSON."""
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    summary = {
        "model": results.get("clustering", {}).get("method"),
        "n_clusters": results.get("clustering", {}).get("n_clusters"),
        "evaluation": results.get("evaluation"),
        "topics": results.get("topics"),
    }

    with output_file.open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    logger.info("Theme analysis summary saved to %s", output_file)


def theme_driver_analysis() -> Dict[str, Any]:
    """Run the theme/driver analysis pipeline.
    
    Supports custom Leiden clustering parameters via environment variables:
    - THEME_LEIDEN_K: Number of neighbors (default: 30)
    - THEME_LEIDEN_RESOLUTION: Resolution parameter (default: 0.7)
    - THEME_LEIDEN_USE_SNN: Use Shared Nearest Neighbor (default: True)
    - THEME_LEIDEN_METRIC: Distance metric (default: cosine)
    - THEME_LEIDEN_RANDOM_STATE: Random seed (default: 42)
    """
    logger.info("Loading theme analysis configuration...")
    azure_config = ConfigManager.load_azure_config()
    theme_config = load_theme_config()

    logger.info("Initializing Azure OpenAI client...")
    client = AzureOpenAI(
        api_key=azure_config.api_key,
        api_version=azure_config.api_version,
        azure_endpoint=azure_config.azure_endpoint,
    )

    logger.info("Initializing ThemeAnalyzer...")
    analyzer = ThemeAnalyzer(client)

    logger.info("Running complete theme analysis...")
    results = analyzer.run_complete_analysis(
        data_folder=theme_config["data_folder"],
        text_column=theme_config["text_column"],
        clustering_method=theme_config["clustering_method"],
        reduce_dimensions=theme_config["reduce_dimensions"],
        **theme_config["clustering_params"],
    )

    logger.info(
        "Theme analysis completed. Found %s clusters.",
        results.get("clustering", {}).get("n_clusters"),
    )

    output_path = theme_config["output_path"]
    if output_path:
        save_theme_results(results, output_path)

    return results


In [0]:
theme_config["clustering_params"]